In [1]:
from langgraph.graph import StateGraph,END,START
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict,Annotated
from langgraph.graph.message import add_messages
from langchain_core.messages import AnyMessage
from langchain_openai import ChatOpenAI

class ChatState(TypedDict):
    response:Annotated[list[AnyMessage],add_messages]

llm=ChatOpenAI(model='gpt-4o-mini')

def node(state:ChatState)->dict:
    message=state["response"]
    response=llm.invoke(message)
    return {"response":[response]}

graph=StateGraph(ChatState)

graph.add_node("node",node)
graph.add_edge(START,"node")
graph.add_edge("node",END)

memory=MemorySaver()
chatbot=graph.compile(checkpointer=memory)

print(f"Chahtbot with memory created")

Chahtbot with memory created


In [2]:
from langchain_core.messages import HumanMessage,AIMessage
config={"configurable": {"thread_id":"user-123"}}

result=chatbot.invoke(
   { "response":[HumanMessage(content="My name is Alice and I love Python programming.")]},config=config
)

print(f"BOT: {result['response'][-1].content}")

result2=chatbot.invoke({"response":HumanMessage(content="What do i love")},config=config)

print(f"BOT: {result2['response'][-1].content}")

result3=chatbot.invoke({"response":HumanMessage(content="Make a Joke about it")},config=config)

print(f"BOT: {result3['response'][-1].content}")


BOT: Hi Alice! It’s great to hear that you love Python programming! Python is a versatile and powerful language, great for everything from web development to data analysis and machine learning. What types of projects or areas are you interested in exploring with Python?
BOT: You mentioned that you love Python programming! If there are other interests or hobbies you have, feel free to share, and I'd be happy to discuss them. What else do you enjoy doing?
BOT: Sure! Here’s a Python-related joke for you:

Why did the Python programmer get kicked out of school?

Because he kept trying to import classmates! 🐍😄

Hope that brings a smile to your face! If you'd like to hear more jokes or anything else, just let me know!


In [7]:
from langchain_core.messages import HumanMessage

config2={"configurable":{"thread_id":"user-1234"}}

result=chatbot.invoke({"response":[HumanMessage(content="My name is Bob and I love JavaScript.")]},config=config2)

print(f"BOT: {result["response"][-1].content}")

result2=chatbot.invoke({"response":[HumanMessage(content="What do i Love")]},config=config2)
print(f"BOT: {result2["response"][-1].content}")

result3=chatbot.invoke({"response":[HumanMessage(content="What do i Love")]},config=config)
print(f"BOT: {result3["response"][-1].content}")

BOT: Hi Bob! It's great to see your enthusiasm for JavaScript. It's such a dynamic and versatile language. If you have any questions, projects in mind, or topics you'd like to discuss related to JavaScript, feel free to share!
BOT: You mentioned that you love JavaScript! If there are other things you're passionate about or topics you'd like to explore, feel free to share!
BOT: You love Python programming! If there's anything specific you'd like to discuss about it—whether it's projects, learning resources, or anything else—let me know!


SQLITE Storage

In [5]:
from langgraph.checkpoint.sqlite import SqliteSaver
import sqlite3
db_path="chatbot.db"
conn=sqlite3.connect(db_path,check_same_thread=False)

persistant_memory=SqliteSaver(conn=conn)
chatbot=graph.compile(checkpointer=persistant_memory)

persistent_config={"configurable":{"thread_id":"1234"}}

result=chatbot.invoke({"response":[HumanMessage(content="Remember Blue is my Favourite colour")]},config=persistent_config)
print(f"BOT: {result["response"][-1].content}")

result2=chatbot.invoke({"response":[HumanMessage(content="What is my fav color")]},config=persistent_config)
print(f"BOT: {result2["response"][-1].content}")

BOT: Absolutely, I’ll remember that blue is your favorite color! If you have anything specific you’d like to discuss about it, feel free to share!
BOT: Your favorite color is blue!


In [9]:
import os
from langgraph.graph import START,StateGraph,END
from langchain_core.messages import HumanMessage,AIMessage,AnyMessage
from langgraph.checkpoint.postgres import PostgresSaver
from langchain_openai import ChatOpenAI
from typing import TypedDict,Annotated
from langgraph.graph.message import add_messages
import psycopg
db_url=os.getenv("SUPABASE_DB_URL")
llm=ChatOpenAI(model='gpt-4o-mini')
class ChatState(TypedDict):
    messages: Annotated[list[AnyMessage],add_messages]

def node(state:ChatState)->dict:
    message=state['messages']
    result=llm.invoke(message)

    return {"messages":[result]}

graph=StateGraph(ChatState)
graph.add_node("node",node)
graph.add_edge(START,"node")
graph.add_edge("node",END)

conn=psycopg.connect(db_url,autocommit=True)

memory=PostgresSaver(conn=conn)
# memory.setup()

chatbot=graph.compile(checkpointer=memory)

config={"configurable":{"thread_id":"user_123"}}

result=chatbot.invoke({"messages":[HumanMessage(content="Remember Blue is my Favourite colour")]},config=config)
print(f"BOT: {result["messages"][-1].content}")

result2=chatbot.invoke({"messages":[HumanMessage(content="What is my Favourite colour")]},config=config)
print(f"BOT: {result2["messages"][-1].content}")

BOT: Absolutely! Blue is your favorite color. If there's anything else you'd like to discuss or ask, just let me know!
BOT: Your favorite color is blue! If there's anything else you'd like to talk about, feel free to ask!


In [11]:
from langgraph.checkpoint.postgres.aio import AsyncPostgresSaver
async with AsyncPostgresSaver.from_conn_string(db_url) as memory:
    await memory.setup()

chatbot=graph.compile(checkpointer=memory)
config={"configurable":{"thread_id":"user_123"}}

result=chatbot.invoke({"messages":[HumanMessage(content="Remember Blue is my Favourite colour")]},config=config)
print(f"BOT: {result["messages"][-1].content}")

result2=chatbot.invoke({"messages":[HumanMessage(content="What is my Favourite colour")]},config=config)
print(f"BOT: {result2["messages"][-1].content}")

In [15]:
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.graph.message import add_messages
from langgraph.checkpoint.sqlite import SqliteSaver

db_url="chatbot2.db"
conn=sqlite3.connect(db_url,check_same_thread=False)

class chatState(TypedDict):
    messages:Annotated[list[AnyMessage],add_messages]

def node(state:ChatState)->dict:
    message=state['messages']
    response=llm.invoke(message)
    return {"messages":[response]}

graph=StateGraph(chatState)
graph.add_node("node",node)
graph.add_edge(START,"node")
graph.add_edge("node",END)

memory=SqliteSaver(conn)
chatbot=graph.compile(checkpointer=memory)

for i in range(3):
    config={"configurable":{"thread_id":f"user-{i}"}}
    chatbot.invoke({"messages":[HumanMessage(content=f"The message for {i}")]},config=config)

state=chatbot.get_state({"configurable":{"thread_id":"user-0"}})
print(f"User state user-0 has {len(state.values['messages'])} messages")

memory.delete_thread('user-0')
state=chatbot.get_state({"configurable":{"thread_id":"user-0"}})
print(f"Current State: user state 0 had now {len(state.values)} messages")
    



User state user-0 has 6 messages
Current State: user state 0 had now 0 messages


In [21]:
import re
import sqlite3
from pathlib import Path

from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import START, END, StateGraph, MessagesState
from langgraph.store.memory import InMemoryStore
from langgraph.store.base import BaseStore
from langgraph.checkpoint.sqlite import SqliteSaver

Path("Memory").mkdir(exist_ok=True)

conn = sqlite3.connect(
    "Memory/chatbot3.db",
    check_same_thread=False,
)

checkpointer = SqliteSaver(conn)
store = InMemoryStore()


def node(state: MessagesState, *, store: BaseStore):
    user_id = "alice"
    namespace = ("users", user_id, "facts")

    # Save a name from the latest user message.
    latest = state["messages"][-1]

    if isinstance(latest, HumanMessage) and isinstance(latest.content, str):
        match = re.search(
            r"\bmy name is\s+([A-Za-z]+)\b",
            latest.content,
            re.IGNORECASE,
        )
        if match:
            user = match.group(1)
            store.put(
                namespace,
                "name",
                {"fact": f"The user's name is {user}"},
            )

    memories = store.search(namespace)
    facts = [m.value.get("fact", "") for m in memories]
    memory_text = "\n".join(facts)

    system = SystemMessage(
        content=f"You are a helpful assistant.\nKnown facts:\n{memory_text}"
    )

    response = llm.invoke([system] + state["messages"])
    return {"messages": [response]}


graph = StateGraph(MessagesState)
graph.add_node("node", node)
graph.add_edge(START, "node")
graph.add_edge("node", END)

app = graph.compile(checkpointer=checkpointer, store=store)

config1 = {"configurable": {"thread_id": "session-1"}}
result = app.invoke(
    {"messages": [HumanMessage(content="Hi! My name is Alice.")]},
    config1,
)
print("Session 1:", result["messages"][-1].content)

config2 = {"configurable": {"thread_id": "session-2"}}
result = app.invoke(
    {"messages": [HumanMessage(content="Do you know my name?")]},
    config2,
)
print("Session 2:", result["messages"][-1].content)

Session 1: Hi Alice! How can I assist you today?
Session 2: Yes, your name is Alice. How can I assist you today?
